# Qwen Original and Full Retraining Results

This notebook compares the predictive utility of the **Original Qwen classifier** with five **Full Retraining** reference models.

Full Retraining is the reference model obtained after removing the records covered by a deletion request from the training data and training again from the original pretrained Qwen base.

No machine unlearning, model loading, training or inference is performed here. The notebook only reads saved experiment artefacts. These reference results will later support the evaluation of approximate unlearning methods.

## 1. Imports and paths

Only the libraries needed to locate and read the saved results are imported.

In [4]:
from pathlib import Path
import json
import pandas as pd

In [5]:
# Find final_submission whether the notebook starts from the repository root or a subfolder.
search_starts = [Path.cwd(), *Path.cwd().parents]
final_submission_candidates = []

for start in search_starts:
    direct = start if start.name == "final_submission" else start / "code" / "final_submission"
    if (direct / "notebooks").is_dir() and (direct / "results").is_dir():
        final_submission_candidates.append(direct.resolve())

if not final_submission_candidates:
    raise FileNotFoundError(
        "Could not locate code/final_submission from the current working directory."
    )

FINAL_SUBMISSION = final_submission_candidates[0]
QWEN_RESULTS = FINAL_SUBMISSION / "results" / "qwen"
FULL_RETRAINING_DIR = QWEN_RESULTS / "full_retraining"

print(f"final_submission: {FINAL_SUBMISSION}")
print(f"Qwen results:     {QWEN_RESULTS}")

final_submission: /Users/niamhhughes/Desktop/Desktop - Niamh’s MacBook Pro/QUB/Research Proj/code/final_submission
Qwen results:     /Users/niamhhughes/Desktop/Desktop - Niamh’s MacBook Pro/QUB/Research Proj/code/final_submission/results/qwen


## 2. Load the Original Qwen baseline

The authoritative baseline is the final learning-rate `1e-5` run selected at epoch 6. Its retained test metrics must be present in the canonical Original Qwen results folder. Older exploratory `2e-5` runs are deliberately not searched as fallbacks.

In [6]:
BASELINE_METRICS_PATH = QWEN_RESULTS / "original" / "retained_test_metrics.csv"
BASELINE_PROVENANCE_PATH = QWEN_RESULTS / "original" / "PROVENANCE.json"

missing_baseline_files = [
    path for path in (BASELINE_METRICS_PATH, BASELINE_PROVENANCE_PATH) if not path.is_file()
]
if missing_baseline_files:
    raise FileNotFoundError(
        "The final Original Qwen artefact is incomplete. Missing:\n  "
        + "\n  ".join(str(path) for path in missing_baseline_files)
        + "\nExpected the final 1e-5 run selected at epoch 6. "
        "Restore that exported CSV rather than substituting an older 2e-5 run "
        "or manually entering metrics."
    )

with BASELINE_PROVENANCE_PATH.open(encoding="utf-8") as file:
    baseline_provenance = json.load(file)

if baseline_provenance.get("status") != "complete":
    raise ValueError("The Original Qwen provenance does not report a complete run.")
if baseline_provenance.get("learning_rate") != 1e-5:
    raise ValueError("The Original Qwen artefact is not the final 1e-5 run.")
if baseline_provenance.get("selected_epoch") != 6:
    raise ValueError("The Original Qwen artefact was not selected at epoch 6.")

baseline_metrics = pd.read_csv(BASELINE_METRICS_PATH)
if len(baseline_metrics) != 1:
    raise ValueError(
        f"Expected one baseline metrics row in {BASELINE_METRICS_PATH}, "
        f"but found {len(baseline_metrics)}."
    )

print(f"Loaded final Original Qwen baseline from:\n  {BASELINE_METRICS_PATH}")
print(
    f"Verified run {baseline_provenance['source_run_id']}: "
    f"learning rate {baseline_provenance['learning_rate']}, "
    f"selected epoch {baseline_provenance['selected_epoch']}."
)
baseline_metrics

Loaded final Original Qwen baseline from:
  /Users/niamhhughes/Desktop/Desktop - Niamh’s MacBook Pro/QUB/Research Proj/code/final_submission/results/qwen/original/retained_test_metrics.csv
Verified run 20260829T151430Z: learning rate 1e-05, selected epoch 6.


,n,positive_count,prevalence,threshold,pr_auc,balanced_accuracy,binary_cross_entropy,f1,auroc,precision,recall,specificity,true_negative,false_positive,false_negative,true_positive
0,8988,665,0.073988,0.55,0.1762,0.604858,0.432329,0.232435,0.728721,0.179153,0.330827,0.87889,7315,1008,445,220


## 3. Load and validate the Full Retraining results

Each scenario contributes its saved retained-test metrics. The completion record and training history are also read so that incomplete or incorrectly assembled runs cannot enter the comparison.

In [7]:
scenarios = [
    ("Recipient Withdrawal", "recipient_withdrawal", 426),
    ("Donor Withdrawal", "donor_withdrawal", 1_992),
    ("Invalid Consent", "invalid_consent", 4_148),
    ("Hospital Removal", "hospital_removal", 4_314),
    ("Retention Expiry", "retention_expiry", 6_262),
]

required_metric_columns = [
    "pr_auc", "balanced_accuracy", "binary_cross_entropy", "f1",
    "auroc", "precision", "recall", "specificity",
]

In [8]:
full_retraining_results = {}
validation_rows = []

for label, folder_name, expected_forget_rows in scenarios:
    scenario_dir = FULL_RETRAINING_DIR / folder_name
    metrics_path = scenario_dir / "retained_test_metrics.csv"
    completion_path = scenario_dir / "COMPLETE.json"
    history_path = scenario_dir / "training_history.csv"

    missing = [path for path in (metrics_path, completion_path, history_path) if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            f"Missing saved artefact(s) for {label}: "
            + ", ".join(str(path) for path in missing)
        )

    with completion_path.open(encoding="utf-8") as file:
        completion = json.load(file)
    history = pd.read_csv(history_path)
    metrics = pd.read_csv(metrics_path)

    if completion.get("status") != "complete":
        raise ValueError(f"{label} has status {completion.get('status')!r}, not 'complete'.")
    if completion.get("training_forget_rows") != expected_forget_rows:
        raise ValueError(
            f"{label} reports {completion.get('training_forget_rows')} training forget rows; "
            f"expected {expected_forget_rows}."
        )
    if history.empty:
        raise ValueError(f"Training history is empty for {label}.")
    if len(metrics) != 1:
        raise ValueError(f"Expected one retained-test metrics row for {label}; found {len(metrics)}.")

    missing_columns = [column for column in required_metric_columns if column not in metrics.columns]
    if missing_columns:
        raise ValueError(f"{label} metrics are missing columns: {missing_columns}")

    full_retraining_results[label] = metrics.iloc[0]
    validation_rows.append({
        "Scenario": label,
        "Status": completion["status"],
        "Forget Rows": completion["training_forget_rows"],
        "History Epochs": len(history),
        "Metrics File": str(metrics_path.relative_to(FINAL_SUBMISSION)),
    })

validation = pd.DataFrame(validation_rows)
validation

,Scenario,Status,Forget Rows,History Epochs,Metrics File
0,Recipient Withdrawal,complete,426,10,results/qwen/full_retraining/recipient_withdra...
1,Donor Withdrawal,complete,1992,8,results/qwen/full_retraining/donor_withdrawal/...
2,Invalid Consent,complete,4148,10,results/qwen/full_retraining/invalid_consent/r...
3,Hospital Removal,complete,4314,9,results/qwen/full_retraining/hospital_removal/...
4,Retention Expiry,complete,6262,10,results/qwen/full_retraining/retention_expiry/...


## 4. Main results table

The table uses the metrics already saved by each experiment. In particular, threshold-dependent metrics are **not recomputed**: they retain the frozen Qwen classification threshold used during evaluation.

In [9]:
missing_baseline_columns = [
    column for column in required_metric_columns if column not in baseline_metrics.columns
]
if missing_baseline_columns:
    raise ValueError(
        f"Original Qwen metrics are missing columns: {missing_baseline_columns}"
    )

metric_column_names = {
    "pr_auc": "PR-AUC ↑",
    "balanced_accuracy": "Balanced Accuracy ↑",
    "binary_cross_entropy": "BCE ↓",
    "f1": "F1 ↑",
    "auroc": "AUROC ↑",
    "precision": "Precision ↑",
    "recall": "Recall ↑",
    "specificity": "Specificity ↑",
}

rows = []
baseline_row = {"Model / Scenario": "Original Qwen", "Forget Rows": "—"}
for source, display_name in metric_column_names.items():
    baseline_row[display_name] = baseline_metrics.iloc[0][source]
rows.append(baseline_row)

for label, _, forget_rows in scenarios:
    result_row = {
        "Model / Scenario": f"{label} — Full Retraining",
        "Forget Rows": f"{forget_rows:,}",
    }
    for source, display_name in metric_column_names.items():
        result_row[display_name] = full_retraining_results[label][source]
    rows.append(result_row)

results_table = pd.DataFrame(rows).set_index("Model / Scenario")
metric_display_columns = list(metric_column_names.values())
results_table.style.format({column: "{:.4f}" for column in metric_display_columns})

,Forget Rows,PR-AUC ↑,Balanced Accuracy ↑,BCE ↓,F1 ↑,AUROC ↑,Precision ↑,Recall ↑,Specificity ↑
Model / Scenario,,,,,,,,,
Original Qwen,—,0.1762,0.6049,0.4323,0.2324,0.7287,0.1792,0.3308,0.8789
Recipient Withdrawal — Full Retraining,426,0.1833,0.5681,0.3532,0.2048,0.7332,0.2257,0.1875,0.9488
Donor Withdrawal — Full Retraining,"1,992",0.1753,0.5439,0.3190,0.1581,0.7286,0.2630,0.1131,0.9747
Invalid Consent — Full Retraining,"4,148",0.1725,0.5644,0.3494,0.1919,0.7360,0.1981,0.1860,0.9429
Hospital Removal — Full Retraining,"4,314",0.1833,0.5611,0.3542,0.1932,0.7395,0.2262,0.1686,0.9536
Retention Expiry — Full Retraining,"6,262",0.1615,0.5493,0.3450,0.1677,0.7352,0.2074,0.1408,0.9578


## 5. Metric guide

| Metric | Meaning | Direction |
|---|---|---|
| PR-AUC ↑ | Primary ranking metric for the imbalanced rejection prediction task. | Higher is better. |
| Balanced Accuracy ↑ | Average performance across the positive and negative classes. | Higher is better. |
| BCE ↓ | Measures the quality and calibration of predicted probabilities. | Lower is better. |
| F1 ↑ | Balances precision and recall for the positive class. | Higher is better. |
| AUROC ↑ | Measures ranking ability across classification thresholds. | Higher is better. |
| Precision ↑ | Of cases predicted positive, how many were actually positive. | Higher is better. |
| Recall ↑ | Of actual positive cases, how many the model detected. | Higher is better. |
| Specificity ↑ | Of actual negative cases, how many the model correctly identified. | Higher is better. |